# 06 — SCD Type 6: Hybrid

Schema:

```text
Type 2 -> version rows
Type 3 -> previous value column
Type 1 -> current row contains latest value
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("scd_type_6_hybrid")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

In [2]:
current_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("previous_segment", StringType(), True),
    StructField("valid_from_raw", StringType(), False),
    StructField("valid_to_raw", StringType(), True),
    StructField("is_current", BooleanType(), False),
])

incoming_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("change_date_raw", StringType(), False),
])

current = (
    spark.createDataFrame(
        [
            ("u1", "retail", None, "2026-01-01", None, True),
            ("u2", "retail", None, "2026-01-01", None, True),
        ],
        current_schema,
    )
    .withColumn("valid_from", F.to_date("valid_from_raw"))
    .withColumn("valid_to", F.to_date("valid_to_raw"))
    .drop("valid_from_raw", "valid_to_raw")
)

incoming = (
    spark.createDataFrame(
        [
            ("u1", "premium", "2026-02-01"),
            ("u2", "retail", "2026-02-01"),
            ("u3", "retail", "2026-02-01"),
        ],
        incoming_schema,
    )
    .withColumn("change_date", F.to_date("change_date_raw"))
    .drop("change_date_raw")
)

current.show(truncate=False)
incoming.show(truncate=False)

+-------+-------+----------------+----------+----------+--------+
|user_id|segment|previous_segment|is_current|valid_from|valid_to|
+-------+-------+----------------+----------+----------+--------+
|u1     |retail |NULL            |true      |2026-01-01|NULL    |
|u2     |retail |NULL            |true      |2026-01-01|NULL    |
+-------+-------+----------------+----------+----------+--------+

+-------+-------+-----------+
|user_id|segment|change_date|
+-------+-------+-----------+
|u1     |premium|2026-02-01 |
|u2     |retail |2026-02-01 |
|u3     |retail |2026-02-01 |
+-------+-------+-----------+



In [3]:
joined = (
    incoming.alias("i")
    .join(current.filter("is_current").alias("c"), on="user_id", how="left")
    .select(
        "user_id",
        F.col("i.segment").alias("new_segment"),
        F.col("i.change_date"),
        F.col("c.segment").alias("old_segment"),
        F.col("c.previous_segment").alias("old_previous_segment"),
        F.col("c.valid_from").alias("old_valid_from"),
    )
    .withColumn("is_new_key", F.col("old_segment").isNull())
    .withColumn("is_changed", (~F.col("is_new_key")) & (F.col("new_segment") != F.col("old_segment")))
)

joined.show(truncate=False)

+-------+-----------+-----------+-----------+--------------------+--------------+----------+----------+
|user_id|new_segment|change_date|old_segment|old_previous_segment|old_valid_from|is_new_key|is_changed|
+-------+-----------+-----------+-----------+--------------------+--------------+----------+----------+
|u1     |premium    |2026-02-01 |retail     |NULL                |2026-01-01    |false     |true      |
|u2     |retail     |2026-02-01 |retail     |NULL                |2026-01-01    |false     |false     |
|u3     |retail     |2026-02-01 |NULL       |NULL                |NULL          |true      |false     |
+-------+-----------+-----------+-----------+--------------------+--------------+----------+----------+



In [4]:
expired_rows = (
    joined
    .filter("is_changed")
    .select(
        "user_id",
        F.col("old_segment").alias("segment"),
        F.col("old_previous_segment").alias("previous_segment"),
        F.col("old_valid_from").alias("valid_from"),
        F.date_sub(F.col("change_date"), 1).alias("valid_to"),
        F.lit(False).alias("is_current"),
    )
)

new_current_rows = (
    joined
    .filter("is_changed OR is_new_key")
    .select(
        "user_id",
        F.col("new_segment").alias("segment"),
        F.when(F.col("is_changed"), F.col("old_segment")).otherwise(F.lit(None).cast(StringType())).alias("previous_segment"),
        F.col("change_date").alias("valid_from"),
        F.lit(None).cast(DateType()).alias("valid_to"),
        F.lit(True).alias("is_current"),
    )
)

changed_keys = joined.filter("is_changed").select("user_id")
unaffected_rows = current.join(changed_keys, on="user_id", how="left_anti")

scd6_result = (
    unaffected_rows
    .unionByName(expired_rows)
    .unionByName(new_current_rows)
    .orderBy("user_id", "valid_from")
)

expired_rows.show(truncate=False)
new_current_rows.show(truncate=False)
scd6_result.show(truncate=False)

+-------+-------+----------------+----------+----------+----------+
|user_id|segment|previous_segment|valid_from|valid_to  |is_current|
+-------+-------+----------------+----------+----------+----------+
|u1     |retail |NULL            |2026-01-01|2026-01-31|false     |
+-------+-------+----------------+----------+----------+----------+

+-------+-------+----------------+----------+--------+----------+
|user_id|segment|previous_segment|valid_from|valid_to|is_current|
+-------+-------+----------------+----------+--------+----------+
|u1     |premium|retail          |2026-02-01|NULL    |true      |
|u3     |retail |NULL            |2026-02-01|NULL    |true      |
+-------+-------+----------------+----------+--------+----------+



+-------+-------+----------------+----------+----------+----------+
|user_id|segment|previous_segment|is_current|valid_from|valid_to  |
+-------+-------+----------------+----------+----------+----------+
|u1     |retail |NULL            |false     |2026-01-01|2026-01-31|
|u1     |premium|retail          |true      |2026-02-01|NULL      |
|u2     |retail |NULL            |true      |2026-01-01|NULL      |
|u3     |retail |NULL            |true      |2026-02-01|NULL      |
+-------+-------+----------------+----------+----------+----------+



In [5]:
totals = spark.createDataFrame(
    [
        ("current_rows", current.count()),
        ("incoming_rows", incoming.count()),
        ("changed_keys", joined.filter("is_changed").count()),
        ("new_keys", joined.filter("is_new_key").count()),
        ("final_rows", scd6_result.count()),
    ],
    StructType([StructField("metric", StringType(), False), StructField("value", LongType(), False)])
)
totals.show(truncate=False)

+-------------+-----+
|metric       |value|
+-------------+-----+
|current_rows |2    |
|incoming_rows|3    |
|changed_keys |1    |
|new_keys     |1    |
|final_rows   |4    |
+-------------+-----+



In [6]:
spark.stop()